# Restaurant Chatbot — Project Workflow

LLM-powered restaurant chatbot with two tools, SQLite database, Docker container, deployed to AWS EC2.

---

## Architecture overview

```
User / chat UI
      |
      v
LLM agent (decides which tool to call)
      |
      +--------------------------+
      |                          |
      v                          v
Tool 1: Menu & prices     Tool 2: Opening hours
(items, categories,       (Mon-Sun schedule)
 costs)
      |                          |
      +--------------------------+
      |
      v
SQLite database (restaurant.db, two tables)
      |
      v
Docker container (Python app + SQLite file bundled)
      |
      v
AWS EC2 instance
```

---

## Stage 1 — Project structure

```
restaurant-bot/
├── app.py              # main LLM loop + tool dispatcher
├── tools/
│   ├── menu_tool.py    # Tool 1
│   └── hours_tool.py   # Tool 2
├── db/
│   ├── seed.py         # creates and populates the SQLite DB
│   └── restaurant.db   # generated file (gitignore this or commit it)
├── requirements.txt
└── Dockerfile
```

---

## Stage 2 — SQLite database design

Single `restaurant.db` file with two independent tables (no foreign keys needed — menu items aren't tied to specific days).

### `menu` table

| Column | Type | Purpose |
|---|---|---|
| `id` | `INTEGER PRIMARY KEY` | Auto-incremented row ID |
| `category` | `TEXT` | "Starters", "Mains", "Desserts", "Drinks" |
| `item_name` | `TEXT` | Dish name, e.g. "Grilled Sea Bass" |
| `description` | `TEXT` | One-line description for the LLM to use in replies |
| `price` | `REAL` | Numeric price, e.g. `14.50` |
| `available` | `INTEGER` | `1` = on menu, `0` = temporarily off |

### `opening_hours` table

| Column | Type | Purpose |
|---|---|---|
| `id` | `INTEGER PRIMARY KEY` | Auto-incremented row ID |
| `day` | `TEXT` | Full name: "Monday", "Tuesday", etc. |
| `opens` | `TEXT` | `HH:MM` format, e.g. `"11:00"` |
| `closes` | `TEXT` | `HH:MM` format, e.g. `"23:30"` |
| `note` | `TEXT` | Optional, e.g. "Kitchen closes 30 min early" |

### Suggested seed data — hours

```
Monday    | 11:00 | 21:00 | NULL
Tuesday   | 11:00 | 21:00 | NULL
Wednesday | 11:00 | 21:00 | NULL
Thursday  | 11:00 | 21:00 | NULL
Friday    | 11:00 | 23:30 | Kitchen closes 30 min early
Saturday  | 11:00 | 23:30 | NULL
Sunday    | 10:00 | 20:00 | Brunch menu until 14:00
```

### Suggested seed data — menu

```
Starters | Garlic Bread      | Toasted sourdough with garlic butter  | 4.50  | 1
Starters | Soup of the Day   | Ask your server for today's selection | 6.00  | 1
Mains    | Grilled Sea Bass  | With lemon butter and seasonal veg    | 18.50 | 1
Mains    | Mushroom Risotto  | Parmesan, truffle oil                 | 14.00 | 1
Desserts | Chocolate Fondant | Warm, with vanilla ice cream          | 7.50  | 1
Drinks   | House Red (175ml) | Rioja                                 | 5.50  | 1
```

---

## Stage 3 — The two LLM tools

Each tool is a Python function that queries SQLite and returns structured data (dict or list of dicts) back to the LLM.

**Tool 1 — `get_menu(category=None)`**
- Queries the `menu` table, filters by `available = 1`
- Optional `category` filter
- Triggered by: "what's on the menu?", "how much is the pasta?", "what desserts do you have?"

**Tool 2 — `get_opening_hours(day=None)`**
- Queries `opening_hours`
- Optional `day` filter, otherwise returns the full week
- Triggered by: "are you open Sunday?", "what time do you close Friday?"

Both tools are registered with the LLM as function-calling tools. The LLM decides which to invoke and with what arguments — routing is never hardcoded.

---

## Stage 4 — LLM agent loop (`app.py`)

1. Receive user message
2. Send to LLM with both tools defined in the `tools` parameter
3. If LLM returns a `tool_use` block → call the relevant Python function → send the result back as a `tool_result`
4. LLM composes a natural language reply using the data
5. Stream or return the reply to the user

Use a `while True` loop with a growing `messages` list to maintain conversational context.

---

## Stage 5 — Dockerfile

1. Base image: `python:3.11-slim`
2. Set working directory to `/app`
3. Copy `requirements.txt` and install dependencies
4. Copy the full project into the container
5. Run `seed.py` at build time to generate `restaurant.db` (baked into the image)
6. Expose the app's port (e.g. 8000 for FastAPI)
7. Entrypoint: `python app.py`

---

## Stage 6 — EC2 deployment workflow

**Local machine:**
1. `docker build -t restaurant-bot .`
2. Tag for ECR or Docker Hub
3. Push the image

**EC2 instance:**
1. Launch instance (Amazon Linux 2 or Ubuntu, `t3.micro` is enough)
2. Install Docker
3. Pull the image from the registry
4. Run: `docker run -d -p 8000:8000 restaurant-bot`
5. Open the port in the EC2 security group

**Secrets handling:**
- Store the LLM API key as an EC2 environment variable or in AWS Secrets Manager — never bake it into the image
- Inject at runtime with `docker run --env-file .env`
- Optional: put Nginx in front for HTTPS

---

## Decision summary

| Question | Answer |
|---|---|
| Database | SQLite, single `.db` file, two tables |
| Tools | Menu & prices, Opening hours |
| Deployment | Docker image with DB baked in |
| Hosting | AWS EC2, any size (`t3.micro` works) |
| Secrets | Env vars at runtime, not in image |

---

## Next steps (not yet built)

- [ ] `seed.py` — DB creation + seeding script
- [ ] `menu_tool.py` / `hours_tool.py` — tool function implementations
- [ ] Tool schema definitions for LLM function-calling
- [ ] `app.py` — agent loop
- [ ] `Dockerfile` + `requirements.txt`
- [ ] EC2 provisioning + security group config


What actually happens
When Docker builds your image, it takes a snapshot of your project files. If restaurant.db existed on your laptop at build time and wasn't in .dockerignore, it would get baked into the image — meaning every container would start with that old database including any old reservations, test data, or mistakes.


By excluding it in .dockerignore:

The image is built without a database file
When the container starts and app.py runs, init_db() is called
init_db() sees no database exists, so it creates a fresh one and seeds it with your menu and opening hours
The reservations table starts empty — clean slate

In [1]:


import os
import re
import sqlite3
import getpass
from operator import itemgetter
from typing import List, Optional



import pandas as pd
from openai import OpenAI
from tavily import TavilyClient
from pydantic import BaseModel, Field



from langchain.agents import create_agent
from langchain.tools import tool

from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
)

from langchain_core.tools import tool

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)

from langchain_core.output_parsers import StrOutputParser

from langchain_core.runnables.history import RunnableWithMessageHistory



from langgraph.prebuilt import (
    create_react_agent,
    ToolNode,
)

from langgraph.graph import (
    StateGraph,
    MessagesState,
    END,
)

from langgraph.checkpoint.memory import MemorySaver



025066244860.dkr.ecr.us-east-1.amazonaws.com/restaurant-bot

/opt/anaconda3/envs/tf-metal/lib/python3.11/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
#OPEN AI  API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI

In [4]:
# Add this — writes the key into a .env file automatically
with open(".env", "w") as f:
    f.write(f"OPENAI_API_KEY={os.environ.get('OPENAI_API_KEY')}\n")

print(".env file created!")

.env file created!


In [5]:




"""
menu_data
Restaurant menu as a nested dictionary.
Category -> list of items, each item a dict with name, description, price.
"""

menu = {
    "breakfast": [
        {"item_name": "Full English Breakfast", "description": "Eggs, bacon, sausage, beans, mushrooms, tomato and toast", "price": 11.50},
        {"item_name": "Eggs Benedict", "description": "Poached eggs, ham and hollandaise on an English muffin", "price": 9.50},
        {"item_name": "Avocado Toast", "description": "Smashed avocado on toasted sourdough", "price": 8.50},
        {"item_name": "Pancake Stack", "description": "Fluffy pancakes with maple syrup and berries", "price": 8.00},
        {"item_name": "French Toast", "description": "Brioche toast served with cinnamon and syrup", "price": 8.50},
        {"item_name": "Breakfast Burrito", "description": "Eggs, sausage, cheese and salsa wrapped in a tortilla", "price": 9.00},
        {"item_name": "Omelette", "description": "Three-egg omelette with cheese, ham or vegetables", "price": 8.50},
        {"item_name": "Greek Yogurt & Granola", "description": "Greek yogurt with granola and seasonal fruit", "price": 6.00},
    ],

    "starters": [
        {"item_name": "Garlic Bread", "description": "Toasted sourdough with garlic butter", "price": 4.50},
        {"item_name": "Soup of the Day", "description": "Ask your server for today's selection", "price": 6.00},
        {"item_name": "Bruschetta", "description": "Grilled bread, tomato, basil, olive oil", "price": 6.50},
        {"item_name": "Calamari", "description": "Lightly fried, served with lemon aioli", "price": 8.50},
        {"item_name": "Chicken Wings", "description": "Crispy wings with BBQ sauce", "price": 7.50},
        {"item_name": "Mozzarella Sticks", "description": "Served with marinara dip", "price": 6.50},
        {"item_name": "Prawn Cocktail", "description": "Classic prawns with Marie Rose sauce", "price": 8.00},
        {"item_name": "Stuffed Mushrooms", "description": "Filled with garlic cream cheese", "price": 7.00},
    ],

    "mains": [
        {"item_name": "Grilled Sea Bass", "description": "With lemon butter and seasonal vegetables", "price": 18.50},
        {"item_name": "Mushroom Risotto", "description": "Arborio rice, parmesan, truffle oil", "price": 14.00},
        {"item_name": "Margherita Pizza", "description": "San Marzano tomato, mozzarella, basil", "price": 12.50},
        {"item_name": "Grilled Ribeye Steak", "description": "10oz, served with fries and peppercorn sauce", "price": 24.00},
        {"item_name": "Chicken Parmigiana", "description": "Breaded chicken breast, marinara, mozzarella", "price": 16.50},
        {"item_name": "Vegetable Curry", "description": "Seasonal vegetables in a coconut curry, served with rice", "price": 13.50},
        {"item_name": "BBQ Bacon Burger", "description": "Beef burger with bacon, cheese and fries", "price": 15.50},
        {"item_name": "Fish and Chips", "description": "Beer-battered cod with mushy peas and fries", "price": 17.00},
        {"item_name": "Spaghetti Bolognese", "description": "Traditional beef ragu and parmesan", "price": 14.50},
        {"item_name": "Lamb Shank", "description": "Slow-cooked lamb with mashed potatoes", "price": 22.00},
        {"item_name": "Chicken Caesar Salad", "description": "Romaine lettuce, parmesan, croutons", "price": 13.00},
        {"item_name": "Prawn Linguine", "description": "Garlic, chilli and white wine sauce", "price": 18.00},
    ],

    "desserts": [
        {"item_name": "Chocolate Fondant", "description": "Warm, with vanilla ice cream", "price": 7.50},
        {"item_name": "Tiramisu", "description": "Classic Italian, espresso-soaked sponge", "price": 6.50},
        {"item_name": "Cheesecake", "description": "New York style, berry compote", "price": 6.50},
        {"item_name": "Apple Crumble", "description": "Served with warm custard", "price": 6.00},
        {"item_name": "Sticky Toffee Pudding", "description": "Rich sponge with toffee sauce", "price": 7.00},
        {"item_name": "Ice Cream Selection", "description": "Three scoops of your choice", "price": 5.50},
        {"item_name": "Brownie Sundae", "description": "Chocolate brownie with ice cream and sauce", "price": 7.50},
    ],

    "drinks": [
        {"item_name": "House Red (175ml)", "description": "Rioja", "price": 5.50},
        {"item_name": "House White (175ml)", "description": "Sauvignon Blanc", "price": 5.50},
        {"item_name": "Draught Beer (Pint)", "description": "Local lager on tap", "price": 5.00},
        {"item_name": "Sparkling Water", "description": "500ml bottle", "price": 2.50},
        {"item_name": "Espresso", "description": "Double shot", "price": 2.80},
        {"item_name": "Cappuccino", "description": "Espresso with steamed milk foam", "price": 3.50},
        {"item_name": "Latte", "description": "Smooth espresso with steamed milk", "price": 3.80},
        {"item_name": "Orange Juice", "description": "Freshly squeezed", "price": 3.50},
        {"item_name": "Coca-Cola", "description": "330ml bottle", "price": 3.00},
        {"item_name": "Gin & Tonic", "description": "Premium gin with tonic water", "price": 8.50},
    ],
}



In [6]:
opening_hours = {
    "Monday":    {"opens": "12:00", "closes": "21:00", "note": ""},
    "Tuesday":   {"opens": "12:00", "closes": "21:00", "note": ""},
    "Wednesday": {"opens": "12:00", "closes": "21:00", "note": ""},
    "Thursday":  {"opens": "12:00", "closes": "21:00", "note": ""},
    "Friday":    {"opens": "12:00", "closes": "23:30", "note": "Kitchen closes 30 min early"},
    "Saturday":  {"opens": "09:00", "closes": "23:30", "note": "Breakfast served until 12:00"},
    "Sunday":    {"opens": "09:00", "closes": "20:00", "note": "Brunch menu until 14:00"},
}

In [7]:
MODEL = "gpt-4.1-nano"

llm = ChatOpenAI(temperature=0, model_name=MODEL)

## CREATE  DATABASE

In [8]:

DB = "restaurant_database.db"

In [9]:
opening_hours

{'Monday': {'opens': '12:00', 'closes': '21:00', 'note': ''},
 'Tuesday': {'opens': '12:00', 'closes': '21:00', 'note': ''},
 'Wednesday': {'opens': '12:00', 'closes': '21:00', 'note': ''},
 'Thursday': {'opens': '12:00', 'closes': '21:00', 'note': ''},
 'Friday': {'opens': '12:00',
  'closes': '23:30',
  'note': 'Kitchen closes 30 min early'},
 'Saturday': {'opens': '09:00',
  'closes': '23:30',
  'note': 'Breakfast served until 12:00'},
 'Sunday': {'opens': '09:00',
  'closes': '20:00',
  'note': 'Brunch menu until 14:00'}}

In [10]:
menu

{'breakfast': [{'item_name': 'Full English Breakfast',
   'description': 'Eggs, bacon, sausage, beans, mushrooms, tomato and toast',
   'price': 11.5},
  {'item_name': 'Eggs Benedict',
   'description': 'Poached eggs, ham and hollandaise on an English muffin',
   'price': 9.5},
  {'item_name': 'Avocado Toast',
   'description': 'Smashed avocado on toasted sourdough',
   'price': 8.5},
  {'item_name': 'Pancake Stack',
   'description': 'Fluffy pancakes with maple syrup and berries',
   'price': 8.0},
  {'item_name': 'French Toast',
   'description': 'Brioche toast served with cinnamon and syrup',
   'price': 8.5},
  {'item_name': 'Breakfast Burrito',
   'description': 'Eggs, sausage, cheese and salsa wrapped in a tortilla',
   'price': 9.0},
  {'item_name': 'Omelette',
   'description': 'Three-egg omelette with cheese, ham or vegetables',
   'price': 8.5},
  {'item_name': 'Greek Yogurt & Granola',
   'description': 'Greek yogurt with granola and seasonal fruit',
   'price': 6.0}],
 'sta

### CREATE TABLES AND UPDATE THE TABLE

In [11]:


### cREATE TABLE FOR RESERVATION, OPENING DAYS AND MENU
##opening days will have details about the timing
##menu will contain the category cost,price, food description
##reservation table can be used for cancelling,boooking or checking




with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS opening_hours (days TEXT, Open TEXT, close TEXT, note TEXT)')
    conn.commit()
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE  IF NOT EXISTS menu ( category TEXT, item_name TEXT, description Text, price REAL)')
    conn.commit()




with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS reservation (
            reservation_id  INTEGER PRIMARY KEY AUTOINCREMENT,
            first_name      TEXT,
            last_name       TEXT,
            year            INTEGER,
            month           INTEGER,
            day             INTEGER,
            time            TEXT,
            status          TEXT
        )
    ''')
    conn.commit()

In [12]:
def set_opening_hours(day, open_time, close_time, note):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            '''
            INSERT INTO opening_hours
            (days, Open, close, note)
            VALUES (?, ?, ?, ?)
            ''',
            (day, open_time, close_time, note)
        )
        conn.commit()

In [13]:
opening_hours.keys()

dict_keys(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])

In [14]:
opening_hours['Monday']

{'opens': '12:00', 'closes': '21:00', 'note': ''}

In [15]:

for i in opening_hours.keys():
    Open, close, note = opening_hours[i]['opens'], opening_hours[i]['closes'], opening_hours[i]['note']
    set_opening_hours(i, Open, close, note)

In [16]:
DB

'restaurant_database.db'

In [17]:


with sqlite3.connect("restaurant_database.db") as conn:
    df_hours = pd.read_sql("SELECT * FROM opening_hours", conn)

df_hours

,days,Open,close,note
0,Monday,12:00,21:00,
1,Tuesday,12:00,21:00,
2,Wednesday,12:00,21:00,
3,Thursday,12:00,21:00,
4,Friday,12:00,23:30,Kitchen closes 30 min early
5,Saturday,09:00,23:30,Breakfast served until 12:00
6,Sunday,09:00,20:00,Brunch menu until 14:00
7,Monday,12:00,21:00,
8,Tuesday,12:00,21:00,
9,Wednesday,12:00,21:00,


In [18]:
def set_menu_item(category, item_name, description, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO menu (category, item_name, description, price) VALUES (?, ?, ?, ?)",
            (category, item_name, description, price)
        )
        conn.commit()
 
for category in menu.keys():
    for item in menu[category]:
        set_menu_item(category, item['item_name'], item['description'], item['price'])

In [19]:
with sqlite3.connect("restaurant_database.db") as conn:
    df_menu = pd.read_sql("SELECT * FROM menu", conn)

df_menu.head()

,category,item_name,description,price
0,breakfast,Full English Breakfast,"Eggs, bacon, sausage, beans, mushrooms, tomato...",11.5
1,breakfast,Eggs Benedict,"Poached eggs, ham and hollandaise on an Englis...",9.5
2,breakfast,Avocado Toast,Smashed avocado on toasted sourdough,8.5
3,breakfast,Pancake Stack,Fluffy pancakes with maple syrup and berries,8.0
4,breakfast,French Toast,Brioche toast served with cinnamon and syrup,8.5


In [20]:
##functions that allow me access the database


# ── Menu 
def get_details_category_all(category):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM menu WHERE category = ?', (category.lower(),))
        results = cursor.fetchall()
        return [
            {"item_name": row[1], "description": row[2], "price": row[3]}
            for row in results
        ]
def get_details_opening_time_all():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM opening_hours')
        results = cursor.fetchall()
        return [
            {"day": row[0], "opens": row[1], "closes": row[2], "note": row[3]}
            for row in results
        ]

In [21]:
print(get_details_category_all('breakfast'))

[{'item_name': 'Full English Breakfast', 'description': 'Eggs, bacon, sausage, beans, mushrooms, tomato and toast', 'price': 11.5}, {'item_name': 'Eggs Benedict', 'description': 'Poached eggs, ham and hollandaise on an English muffin', 'price': 9.5}, {'item_name': 'Avocado Toast', 'description': 'Smashed avocado on toasted sourdough', 'price': 8.5}, {'item_name': 'Pancake Stack', 'description': 'Fluffy pancakes with maple syrup and berries', 'price': 8.0}, {'item_name': 'French Toast', 'description': 'Brioche toast served with cinnamon and syrup', 'price': 8.5}, {'item_name': 'Breakfast Burrito', 'description': 'Eggs, sausage, cheese and salsa wrapped in a tortilla', 'price': 9.0}, {'item_name': 'Omelette', 'description': 'Three-egg omelette with cheese, ham or vegetables', 'price': 8.5}, {'item_name': 'Greek Yogurt & Granola', 'description': 'Greek yogurt with granola and seasonal fruit', 'price': 6.0}, {'item_name': 'Full English Breakfast', 'description': 'Eggs, bacon, sausage, bean

In [22]:
get_details_opening_time_all()


[{'day': 'Monday', 'opens': '12:00', 'closes': '21:00', 'note': ''},
 {'day': 'Tuesday', 'opens': '12:00', 'closes': '21:00', 'note': ''},
 {'day': 'Wednesday', 'opens': '12:00', 'closes': '21:00', 'note': ''},
 {'day': 'Thursday', 'opens': '12:00', 'closes': '21:00', 'note': ''},
 {'day': 'Friday',
  'opens': '12:00',
  'closes': '23:30',
  'note': 'Kitchen closes 30 min early'},
 {'day': 'Saturday',
  'opens': '09:00',
  'closes': '23:30',
  'note': 'Breakfast served until 12:00'},
 {'day': 'Sunday',
  'opens': '09:00',
  'closes': '20:00',
  'note': 'Brunch menu until 14:00'},
 {'day': 'Monday', 'opens': '12:00', 'closes': '21:00', 'note': ''},
 {'day': 'Tuesday', 'opens': '12:00', 'closes': '21:00', 'note': ''},
 {'day': 'Wednesday', 'opens': '12:00', 'closes': '21:00', 'note': ''},
 {'day': 'Thursday', 'opens': '12:00', 'closes': '21:00', 'note': ''},
 {'day': 'Friday',
  'opens': '12:00',
  'closes': '23:30',
  'note': 'Kitchen closes 30 min early'},
 {'day': 'Saturday',
  'opens

In [23]:
# ── Reservations ───────────────────────────────────────────────────────────
def create_reservation(first_name, last_name, year, month, day, time):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            '''
            INSERT INTO reservation (first_name, last_name, year, month, day, time, status)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            ''',
            (first_name.lower(), last_name.lower(), year, month, day, time, "confirmed")
        )
        conn.commit()
        reservation_id = cursor.lastrowid
        return reservation_id

def check_reservation_db(first_name, last_name):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'SELECT * FROM reservation WHERE first_name = ? AND last_name = ?',
            (first_name.lower(), last_name.lower())
        )
        results = cursor.fetchall()
        if not results:
            return []
        return [
            {
                "reservation_id": row[0],
                "first_name":     row[1],
                "last_name":      row[2],
                "year":           row[3],
                "month":          row[4],
                "day":            row[5],
                "time":           row[6],
                "status":         row[7]
            }
            for row in results
        ]

def cancel_reservation_db(first_name, last_name):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'UPDATE reservation SET status = ? WHERE first_name = ? AND last_name = ?',
            ("cancelled", first_name.lower(), last_name.lower())
        )
        conn.commit()
        if cursor.rowcount == 0:
            return None
        cursor.execute(
            'SELECT * FROM reservation WHERE first_name = ? AND last_name = ?',
            (first_name.lower(), last_name.lower())
        )
        row = cursor.fetchone()
        return {
            "reservation_id": row[0],
            "first_name":     row[1],
            "last_name":      row[2],
            "year":           row[3],
            "month":          row[4],
            "day":            row[5],
            "time":           row[6],
            "status":         row[7]
        }

## SCHEMA FOR PYDANTIC OUTPUT

In [24]:
# ── Menu schemas ───────────────────────────────────────────────────────────
class FoodDrinkResult(BaseModel):
    """The details about the food such as name and price."""
    item_name: str = Field(description="name of food or drink")
    price:     str = Field(description="Price as a string, e.g. '$14.50'")

class FoodAgentResponse(BaseModel):
    """Schema for agent response with description, price, and name."""
    answer:      str                 = Field(description="The agent's answer to the query")
    cost:        List[FoodDrinkResult] = Field(default_factory=list, description="the food name and price")
    description: str                 = Field(description="food details")

    
# ── Tool 1: Menu prices ────────────────────────────────────────────────────
@tool
def menu_prices(category: Optional[str] = None) -> FoodAgentResponse:
    """
    Use this tool to find restaurant menu items, prices, and descriptions.
    The guest states what they want and use this tool to provide varieties
    of what is offered based on each category.
    Use this for any question about menu prices, food details, or what's available.

    Args:
        category: Optional. One of ""breakfast,"starters", "mains", "desserts", "drinks".
                  If omitted, returns the full menu.

    Returns a FoodAgentResponse with:
      - answer:      plain-English summary
      - cost:        list of items with name and price
      - description: combined description text for the matching items
    """
    valid_categories = ["breakfast","starters", "mains", "desserts", "drinks"]
    if category:
        category = category.lower()
        if category not in valid_categories:
            return FoodAgentResponse(
                answer=f"Invalid category. Please choose from: {', '.join(valid_categories)}",
                cost=[],
                description=""
            )
        items = get_details_category_all(category)
        if not items:
            return FoodAgentResponse(
                answer=f"No items found for {category}.",
                cost=[],
                description=""
            )
        return FoodAgentResponse(
            answer=f"Here are the available {category}.",
            cost=[FoodDrinkResult(item_name=item["item_name"], price=str(item["price"])) for item in items],
            description=" ".join(item["description"] for item in items)
        )
    return FoodAgentResponse(
        answer=f"Please specify a category: {', '.join(valid_categories)}",
        cost=[],
        description=""
    )

In [25]:
class OpeningHoursResult(BaseModel):
    """Opening hours details for a single day."""
    day:    str = Field(description="day of the week")
    opens:  str = Field(description="opening time, e.g. '12:00'")
    closes: str = Field(description="closing time, e.g. '21:00'")
    note:   str = Field(description="any special note for the day")

class OpeningHoursResponse(BaseModel):
    """Schema for agent response with opening hours."""
    answer: str                      = Field(description="The agent's answer to the query")
    hours:  List[OpeningHoursResult] = Field(default_factory=list, description="opening hours for the requested day(s)")





# Tool 2: Opening hours 
@tool
def opening_hours_tool(day: Optional[str] = None) -> OpeningHoursResponse:
    """
    Use this tool to answer customer inquiries about restaurant opening hours,
    closing times, and any special notes such as kitchen closing early,
    breakfast, or brunch availability.

    Args:
        day: Optional. One of "Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday".
             If omitted, returns hours for the full week.

    Returns an OpeningHoursResponse with:
      - answer: plain-English summary
      - hours:  list of day(s) with opening time, closing time, and note
    """
    all_hours = get_details_opening_time_all()

    if day:
        day = day.capitalize()
        match = [r for r in all_hours if r["day"] == day]
        if not match:
            return OpeningHoursResponse(
                answer=f"No opening hours found for {day}.",
                hours=[]
            )
        info = match[0]
        return OpeningHoursResponse(
            answer=f"On {day}, we're open from {info['opens']} to {info['closes']}.",
            hours=[
                OpeningHoursResult(
                    day=info["day"],
                    opens=info["opens"],
                    closes=info["closes"],
                    note=info["note"]
                )
            ]
        )

    return OpeningHoursResponse(
        answer="Here are our opening hours for the week.",
        hours=[
            OpeningHoursResult(
                day=r["day"],
                opens=r["opens"],
                closes=r["closes"],
                note=r["note"]
            )
            for r in all_hours
        ]
    )

In [26]:

class ReservationResult(BaseModel):
    """The details about a reservation."""
    reservation_id: int = Field(description="Unique booking reference number")
    first_name:     str = Field(description="First name of the guest")
    last_name:      str = Field(description="Last name of the guest")
    year:           int = Field(description="Year of the reservation, e.g. 2026")
    month:          int = Field(description="Month of the reservation, e.g. 7")
    day:            int = Field(description="Day of the reservation, e.g. 5")
    time:           str = Field(description="Reservation time, e.g. '19:00'")
    status:         str = Field(description="Status: 'confirmed' or 'cancelled'")

class ReservationAgentResponse(BaseModel):
    """Schema for agent response for booking, cancelling, and checking reservations."""
    answer:       str                      = Field(description="The agent's plain-English answer to the query")
    reservations: List[ReservationResult]  = Field(default_factory=list, description="List of reservation details")


In [27]:
@tool
def table_reservation(first_name: str, last_name: str, year: int, month: int, day: int, time: str) -> ReservationAgentResponse:
    """
    Use this tool to create a reservation for a guest.
    You MUST collect ALL of the following before calling this tool:
    first name, last name, year, month, day, and time.
    Do NOT assume any field — ask for each one explicitly.
    Do NOT assume the year — always ask the guest for the year.
    Do NOT proceed with only a first name — always ask for the last name.
    If the guest gives a month name (e.g. "July"), convert it to an integer (7).
    If the guest gives a vague time (e.g. "evening"), ask for a specific time in HH:MM format.

    Args:
        first_name: First name of the guest — always ask explicitly, never assume
        last_name:  Last name of the guest — always ask explicitly, never assume
        year:       Year of the reservation, e.g. 2026 — always ask, never assume
        month:      Month as an integer, e.g. 7 for July
        day:        Day as an integer, e.g. 5
        time:       Time as a string in HH:MM format, e.g. "19:00"

    Returns a ReservationAgentResponse with:
      - answer:        confirmation message with the reservation ID
      - reservations:  list with the full booking details
    """
    reservation_id = create_reservation(first_name, last_name, year, month, day, time)
    return ReservationAgentResponse(
        answer=f"Reservation confirmed! Your booking reference is #{reservation_id}. See you on {day}/{month}/{year} at {time}, {first_name.title()}!",
        reservations=[
            ReservationResult(
                reservation_id=reservation_id,
                first_name=first_name,
                last_name=last_name,
                year=year,
                month=month,
                day=day,
                time=time,
                status="confirmed"
            )
        ]
    )

# ── Tool 4: 
@tool
def check_reservation(first_name: str, last_name: str) -> ReservationAgentResponse:
    """
    Use this tool to check the status of an existing reservation for a guest.
    You MUST collect both first name AND last name before calling this tool.
    Do NOT assume or proceed with only one name — always ask for the last name explicitly.
    Do NOT assume the guest's last name from context or previous messages.
    Return the status of the reservation, 'confirmed' or 'cancelled'

    Args:
        first_name: First name of the guest — always ask explicitly, never assume
        last_name:  Last name of the guest — always ask explicitly, never assume

    Returns a ReservationAgentResponse with:
      - answer:        confirmation message with the reservation ID
      - reservations:  list with the full booking details
    """
    results = check_reservation_db(first_name, last_name)
    if not results:
        return ReservationAgentResponse(
            answer=f"No reservation found for {first_name.title()} {last_name.title()}. Please double check the name.",
            reservations=[]
        )
    return ReservationAgentResponse(
        answer=f"Found {len(results)} reservation(s) for {first_name.title()} {last_name.title()}.",
        reservations=[
            ReservationResult(
                reservation_id=r["reservation_id"],
                first_name=r["first_name"],
                last_name=r["last_name"],
                year=r["year"],
                month=r["month"],
                day=r["day"],
                time=r["time"],
                status=r["status"]
            )
            for r in results
        ]
    )

# ── Tool 5: Cancel reservation ─────────────────────────────────────────────
@tool
def cancel_reservation(first_name: str, last_name: str) -> ReservationAgentResponse:
    """
    Use this tool to cancel an existing reservation for a guest.
    You MUST collect both first name AND last name before calling this tool.
    Do NOT assume or proceed with only one name — always ask for the last name explicitly.
    Do NOT cancel without verbally confirming the full name with the guest first.
    Do NOT assume the cancellation is for the same person as a previous query in the conversation.
    Return a confirmation message of cancellation

    Args:
        first_name: First name of the guest — always ask explicitly, never assume
        last_name:  Last name of the guest — always ask explicitly, never assume

    Returns a ReservationAgentResponse with:
      - answer:        cancellation confirmation message
      - reservations:  list with the updated booking details showing cancelled status
    """
    result = cancel_reservation_db(first_name, last_name)
    if not result:
        return ReservationAgentResponse(
            answer=f"No active reservation found for {first_name.title()} {last_name.title()}. Please double check the name and try again.",
            reservations=[]
        )
    return ReservationAgentResponse(
        answer=f"Reservation for {first_name.title()} {last_name.title()} on {result['day']}/{result['month']}/{result['year']} at {result['time']} has been successfully cancelled.",
        reservations=[
            ReservationResult(
                reservation_id=result["reservation_id"],
                first_name=result["first_name"],
                last_name=result["last_name"],
                year=result["year"],
                month=result["month"],
                day=result["day"],
                time=result["time"],
                status=result["status"]
            )
        ]
    )


In [28]:
tools = [menu_prices, opening_hours_tool, table_reservation, check_reservation, cancel_reservation]

In [39]:
RESTAURANT_SYSTEM = """
You are Theo — the warm, sharp-eyed host of the restaurant, the kind of person
who remembers regulars by their order and treats every guest like they've
been coming for years.

You have access to five tools. Use them smartly:
- menu_prices         → guest asks about menu items, prices, ingredients, or what's available
- opening_hours_tool  → guest asks about hours, whether you're open, or special notes (kitchen closing, brunch, etc.)
- table_reservation   → create a reservation — requires first name, last name, year, month, day, and time
- check_reservation   → return the status of a reservation — requires first name and last name
- cancel_reservation  → cancel an existing reservation — requires first name and last name

Tool rules:
- NEVER guess prices, menu items, or hours. Always use the tool.
- Call the tool first, then weave the result into natural conversation.
- If the guest doesn't specify a category or day, call the tool with no
  argument to give them the full picture rather than asking unnecessarily.
- You can call multiple tools in one turn if needed.

Reservation rules — follow these without exception:
- NEVER call table_reservation without ALL six fields: first name, last name, year, month, day, and time.
- NEVER assume the year — always ask the guest explicitly.
- NEVER assume the month  - always ask the guest explicitly
- ALWAYS ASK FOR THE MONTH, DAY AND TIME
- NEVER assume the date - always ask the guest explicitly
- NEVER assume a time, always ask the guest explicitly
- NEVER proceed with only a first name — always ask for the last name before calling any reservation tool.
- ALWAYS ASK FOR THE MONTH, DAY AND TIME
- NEVER assume a last name from context or earlier in the conversation.
- If the guest gives a month name like "July", convert it to 7 before calling the tool.
- If the guest gives a vague time like "evening" or "around 7", ask for the exact time in HH:MM format.
- Before cancelling, always confirm the full name back to the guest and ask them to confirm before calling the tool.
- If a reservation is not found, tell the guest warmly and ask them to double check the name.

How to sound like Theo:
- Open with something inviting — the smell of the kitchen, the buzz of the
  room, the specials board, the clatter of plates.
  Never "Great choice!" or "Absolutely!" — banned.
- Weave tool results into flowing prose. Never dump raw prices, items, or
  hours as a list or table.
- Be honest. If the kitchen's about to close, say so. If a dish isn't great
  that night, say so. If hours are unusual on a given day, mention it.
- 3–6 sentences of flowing prose. No bullets. No headers. No lists.
- Close with a recommendation or a small question to help the guest decide.

Language rule: Always reply in the same language the guest wrote in.
"""

In [40]:
tools

[StructuredTool(name='menu_prices', description='Use this tool to find restaurant menu items, prices, and descriptions.\nThe guest states what they want and use this tool to provide varieties\nof what is offered based on each category.\nUse this for any question about menu prices, food details, or what\'s available.\n\nArgs:\n    category: Optional. One of ""breakfast,"starters", "mains", "desserts", "drinks".\n              If omitted, returns the full menu.\n\nReturns a FoodAgentResponse with:\n  - answer:      plain-English summary\n  - cost:        list of items with name and price\n  - description: combined description text for the matching items', args_schema=<class 'langchain_core.utils.pydantic.menu_prices'>, func=<function menu_prices at 0x168160180>),
 StructuredTool(name='opening_hours_tool', description='Use this tool to answer customer inquiries about restaurant opening hours,\nclosing times, and any special notes such as kitchen closing early,\nbreakfast, or brunch availa

In [41]:
llm

ChatOpenAI(profile={'name': 'GPT-4.1 nano', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x1680eff50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x16812b990>, root_client=<openai.OpenAI object at 0x1676a0290>, root_async_client=<openai.AsyncOpenAI object at 0x16812b4d0>, model_name='gpt-4.1-nano', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('***

In [42]:

llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools)

In [43]:
#
#   [START]
#      │
#      ▼
#  [agent_reason]  ← theo thinks
#      │
#      ▼
#  should_continue?
#      │
#      ├── tool_calls? → [act / tool_node] → back to [agent_reason]
#      │
#      └── no tool_calls? → [END] → final answer

In [44]:

def agent_reason(state: MessagesState) -> MessagesState:
    """theo think — reads history, decides to call tool or respond."""
    response = llm_with_tools.invoke([
        SystemMessage(content=RESTAURANT_SYSTEM),
        *state["messages"]           # ← full conversation history from state
    ])
    return {"messages": [response]}  # ← appends to state, not replaces

# ── Routing ──
def should_continue(state: MessagesState) -> str:
    """Check last message — if tool calls exist, act. Otherwise end."""
    if state["messages"][-1].tool_calls:
        return "act"
    return END

# ── Graph 
graph = StateGraph(MessagesState)
##add the theo prompt
graph.add_node("agent_reason", agent_reason)
##add the tool to the work
graph.add_node("act",          tool_node)

###this is where we start from
graph.set_entry_point("agent_reason")
##from the answere here, where do we go, from agent then we go to should continue, which direction
##should continue act- use a tool or end
graph.add_conditional_edges("agent_reason", should_continue, {"act": "act", END: END})
### if it acts, and not end, it goes back to main system and see if there is a need for response, then back loop agent reason -- > end if no tool is needed
graph.add_edge("act", "agent_reason")

# ── Compile with memory 
app = graph.compile(checkpointer=MemorySaver())



In [45]:
# ── Chat function 
def chat(question: str, session_id: str = "user_001") -> str:
    config   = {"configurable": {"thread_id": session_id}}
    response = app.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config
    )
    return response["messages"][-1].content

In [46]:
chat('whats do you have for breakfast??')

"We've got a lovely selection for breakfast, from hearty Full English Breakfast and Eggs Benedict to lighter options like Greek Yogurt with Granola. If you're craving something sweet, the Pancake Stack and French Toast are always popular. Would you like a recommendation based on your preferences?"

In [47]:
chat('is it possible to make a reservation')

"Absolutely, I'd be happy to help you with a reservation. Could you please tell me your first and last name? Also, I need to know the year, month, day, and the time you'd like to reserve. Do you have a specific date and time in mind?"

In [48]:
chat('gabriel oluwatoyin, 25th and at 11 am')

"Thanks for that, Gabriel. I just need to confirm the year and the month for your reservation. Could you please tell me the year and the month you'd like to book for?"

In [49]:
chat('THE YEAR is 2026 and August')

"Your reservation is all set for August 25th, 2026 at 11:00 in the morning, Gabriel. You're booked under your name, and I look forward to welcoming you then. Would you like any recommendations for what to try while you're here?"

In [50]:
with sqlite3.connect(DB) as conn:
    df_menu = pd.read_sql("SELECT * FROM reservation", conn)

df_menu

,reservation_id,first_name,last_name,year,month,day,time,status
0,1,gabriel,oluwatoyin,2024,4,25,11:00,confirmed
1,2,gabriel,oluwatoyin,2026,8,25,11:00,confirmed


In [51]:
chat('i would like to cancel my reservation')

"Of course, Gabriel. Just to make sure, you're looking to cancel your reservation for August 25th, 2026 at 11:00, correct? Please confirm, and I'll take care of it for you."

In [52]:
chat('yes that reservation')

'Your reservation for August 25th, 2026 at 11:00 has been successfully cancelled, Gabriel. If you need anything else or decide to reschedule, just let me know. Always happy to help!'

In [53]:
with sqlite3.connect(DB) as conn:
    df_menu = pd.read_sql("SELECT * FROM reservation", conn)

df_menu

,reservation_id,first_name,last_name,year,month,day,time,status
0,1,gabriel,oluwatoyin,2024,4,25,11:00,cancelled
1,2,gabriel,oluwatoyin,2026,8,25,11:00,cancelled


## STREAMLIT FILE

In [168]:
%%writefile streamlit_app.py

import os
import sqlite3
import streamlit as st
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY from .env automatically

from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
DB      = "restaurant_database.db"
MODEL   = "gpt-4.1-nano"

# ══════════════════════════════════════════════════════════════════════════════
# DATA
# ══════════════════════════════════════════════════════════════════════════════
menu = {
    "breakfast": [
        {"item_name": "Full English Breakfast", "description": "Eggs, bacon, sausage, beans, mushrooms, tomato and toast", "price": 11.50},
        {"item_name": "Eggs Benedict", "description": "Poached eggs, ham and hollandaise on an English muffin", "price": 9.50},
        {"item_name": "Avocado Toast", "description": "Smashed avocado on toasted sourdough", "price": 8.50},
        {"item_name": "Pancake Stack", "description": "Fluffy pancakes with maple syrup and berries", "price": 8.00},
        {"item_name": "French Toast", "description": "Brioche toast served with cinnamon and syrup", "price": 8.50},
        {"item_name": "Breakfast Burrito", "description": "Eggs, sausage, cheese and salsa wrapped in a tortilla", "price": 9.00},
        {"item_name": "Omelette", "description": "Three-egg omelette with cheese, ham or vegetables", "price": 8.50},
        {"item_name": "Greek Yogurt & Granola", "description": "Greek yogurt with granola and seasonal fruit", "price": 6.00},
    ],
    "starters": [
        {"item_name": "Garlic Bread", "description": "Toasted sourdough with garlic butter", "price": 4.50},
        {"item_name": "Soup of the Day", "description": "Ask your server for today's selection", "price": 6.00},
        {"item_name": "Bruschetta", "description": "Grilled bread, tomato, basil, olive oil", "price": 6.50},
        {"item_name": "Calamari", "description": "Lightly fried, served with lemon aioli", "price": 8.50},
        {"item_name": "Chicken Wings", "description": "Crispy wings with BBQ sauce", "price": 7.50},
        {"item_name": "Mozzarella Sticks", "description": "Served with marinara dip", "price": 6.50},
        {"item_name": "Prawn Cocktail", "description": "Classic prawns with Marie Rose sauce", "price": 8.00},
        {"item_name": "Stuffed Mushrooms", "description": "Filled with garlic cream cheese", "price": 7.00},
    ],
    "mains": [
        {"item_name": "Grilled Sea Bass", "description": "With lemon butter and seasonal vegetables", "price": 18.50},
        {"item_name": "Mushroom Risotto", "description": "Arborio rice, parmesan, truffle oil", "price": 14.00},
        {"item_name": "Margherita Pizza", "description": "San Marzano tomato, mozzarella, basil", "price": 12.50},
        {"item_name": "Grilled Ribeye Steak", "description": "10oz, served with fries and peppercorn sauce", "price": 24.00},
        {"item_name": "Chicken Parmigiana", "description": "Breaded chicken breast, marinara, mozzarella", "price": 16.50},
        {"item_name": "Vegetable Curry", "description": "Seasonal vegetables in a coconut curry, served with rice", "price": 13.50},
        {"item_name": "BBQ Bacon Burger", "description": "Beef burger with bacon, cheese and fries", "price": 15.50},
        {"item_name": "Fish and Chips", "description": "Beer-battered cod with mushy peas and fries", "price": 17.00},
        {"item_name": "Spaghetti Bolognese", "description": "Traditional beef ragu and parmesan", "price": 14.50},
        {"item_name": "Lamb Shank", "description": "Slow-cooked lamb with mashed potatoes", "price": 22.00},
        {"item_name": "Chicken Caesar Salad", "description": "Romaine lettuce, parmesan, croutons", "price": 13.00},
        {"item_name": "Prawn Linguine", "description": "Garlic, chilli and white wine sauce", "price": 18.00},
    ],
    "desserts": [
        {"item_name": "Chocolate Fondant", "description": "Warm, with vanilla ice cream", "price": 7.50},
        {"item_name": "Tiramisu", "description": "Classic Italian, espresso-soaked sponge", "price": 6.50},
        {"item_name": "Cheesecake", "description": "New York style, berry compote", "price": 6.50},
        {"item_name": "Apple Crumble", "description": "Served with warm custard", "price": 6.00},
        {"item_name": "Sticky Toffee Pudding", "description": "Rich sponge with toffee sauce", "price": 7.00},
        {"item_name": "Ice Cream Selection", "description": "Three scoops of your choice", "price": 5.50},
        {"item_name": "Brownie Sundae", "description": "Chocolate brownie with ice cream and sauce", "price": 7.50},
    ],
    "drinks": [
        {"item_name": "House Red (175ml)", "description": "Rioja", "price": 5.50},
        {"item_name": "House White (175ml)", "description": "Sauvignon Blanc", "price": 5.50},
        {"item_name": "Draught Beer (Pint)", "description": "Local lager on tap", "price": 5.00},
        {"item_name": "Sparkling Water", "description": "500ml bottle", "price": 2.50},
        {"item_name": "Espresso", "description": "Double shot", "price": 2.80},
        {"item_name": "Cappuccino", "description": "Espresso with steamed milk foam", "price": 3.50},
        {"item_name": "Latte", "description": "Smooth espresso with steamed milk", "price": 3.80},
        {"item_name": "Orange Juice", "description": "Freshly squeezed", "price": 3.50},
        {"item_name": "Coca-Cola", "description": "330ml bottle", "price": 3.00},
        {"item_name": "Gin & Tonic", "description": "Premium gin with tonic water", "price": 8.50},
    ],
}

opening_hours = {
    "Monday":    {"opens": "12:00", "closes": "21:00", "note": ""},
    "Tuesday":   {"opens": "12:00", "closes": "21:00", "note": ""},
    "Wednesday": {"opens": "12:00", "closes": "21:00", "note": ""},
    "Thursday":  {"opens": "12:00", "closes": "21:00", "note": ""},
    "Friday":    {"opens": "12:00", "closes": "23:30", "note": "Kitchen closes 30 min early"},
    "Saturday":  {"opens": "09:00", "closes": "23:30", "note": "Breakfast served until 12:00"},
    "Sunday":    {"opens": "09:00", "closes": "20:00", "note": "Brunch menu until 14:00"},
}

# ══════════════════════════════════════════════════════════════════════════════
# DATABASE SETUP
# ══════════════════════════════════════════════════════════════════════════════
def init_db():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('CREATE TABLE IF NOT EXISTS opening_hours (days TEXT, Open TEXT, close TEXT, note TEXT)')
        cursor.execute('CREATE TABLE IF NOT EXISTS menu (category TEXT, item_name TEXT, description TEXT, price REAL)')
        cursor.execute('''CREATE TABLE IF NOT EXISTS reservation (
            reservation_id INTEGER PRIMARY KEY AUTOINCREMENT,
            first_name TEXT, last_name TEXT,
            year INTEGER, month INTEGER, day INTEGER,
            time TEXT, status TEXT
        )''')
        # seed only if empty
        if cursor.execute("SELECT COUNT(*) FROM opening_hours").fetchone()[0] == 0:
            for day, info in opening_hours.items():
                cursor.execute("INSERT INTO opening_hours (days, Open, close, note) VALUES (?,?,?,?)",
                               (day, info["opens"], info["closes"], info["note"]))
        if cursor.execute("SELECT COUNT(*) FROM menu").fetchone()[0] == 0:
            for category, items in menu.items():
                for item in items:
                    cursor.execute("INSERT INTO menu (category, item_name, description, price) VALUES (?,?,?,?)",
                                   (category, item["item_name"], item["description"], item["price"]))
        conn.commit()

# ══════════════════════════════════════════════════════════════════════════════
# DB FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════
def get_details_category_all(category):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM menu WHERE category = ?', (category.lower(),))
        results = cursor.fetchall()
        return [{"item_name": row[1], "description": row[2], "price": row[3]} for row in results]

def get_details_opening_time_all():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM opening_hours')
        results = cursor.fetchall()
        return [{"day": row[0], "opens": row[1], "closes": row[2], "note": row[3]} for row in results]

def create_reservation(first_name, last_name, year, month, day, time):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO reservation (first_name, last_name, year, month, day, time, status) VALUES (?,?,?,?,?,?,?)',
            (first_name.lower(), last_name.lower(), year, month, day, time, "confirmed")
        )
        conn.commit()
        return cursor.lastrowid

def check_reservation_db(first_name, last_name):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM reservation WHERE first_name = ? AND last_name = ?',
                       (first_name.lower(), last_name.lower()))
        results = cursor.fetchall()
        if not results:
            return []
        return [{"reservation_id": r[0], "first_name": r[1], "last_name": r[2],
                 "year": r[3], "month": r[4], "day": r[5], "time": r[6], "status": r[7]} for r in results]

def cancel_reservation_db(first_name, last_name):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('UPDATE reservation SET status = ? WHERE first_name = ? AND last_name = ?',
                       ("cancelled", first_name.lower(), last_name.lower()))
        conn.commit()
        if cursor.rowcount == 0:
            return None
        cursor.execute('SELECT * FROM reservation WHERE first_name = ? AND last_name = ?',
                       (first_name.lower(), last_name.lower()))
        row = cursor.fetchone()
        return {"reservation_id": row[0], "first_name": row[1], "last_name": row[2],
                "year": row[3], "month": row[4], "day": row[5], "time": row[6], "status": row[7]}

# ══════════════════════════════════════════════════════════════════════════════
# SCHEMAS
# ══════════════════════════════════════════════════════════════════════════════
class FoodDrinkResult(BaseModel):
    item_name: str = Field(description="name of food or drink")
    price: str     = Field(description="Price as a string, e.g. '$14.50'")

class FoodAgentResponse(BaseModel):
    answer:      str                   = Field(description="The agent's answer to the query")
    cost:        List[FoodDrinkResult] = Field(default_factory=list)
    description: str                   = Field(description="food details")

class OpeningHoursResult(BaseModel):
    day:    str = Field(description="day of the week")
    opens:  str = Field(description="opening time")
    closes: str = Field(description="closing time")
    note:   str = Field(description="special note")

class OpeningHoursResponse(BaseModel):
    answer: str                      = Field(description="The agent's answer")
    hours:  List[OpeningHoursResult] = Field(default_factory=list)

class ReservationResult(BaseModel):
    reservation_id: int = Field(description="Unique booking reference number")
    first_name:     str = Field(description="First name of the guest")
    last_name:      str = Field(description="Last name of the guest")
    year:           int = Field(description="Year of the reservation")
    month:          int = Field(description="Month of the reservation")
    day:            int = Field(description="Day of the reservation")
    time:           str = Field(description="Reservation time")
    status:         str = Field(description="'confirmed' or 'cancelled'")

class ReservationAgentResponse(BaseModel):
    answer:       str                     = Field(description="The agent's plain-English answer")
    reservations: List[ReservationResult] = Field(default_factory=list)

# ══════════════════════════════════════════════════════════════════════════════
# TOOLS
# ══════════════════════════════════════════════════════════════════════════════
@tool
def menu_prices(category: Optional[str] = None) -> FoodAgentResponse:
    """
    Use this tool to find restaurant menu items, prices, and descriptions.
    The guest states what they want and use this tool to provide varieties
    of what is offered based on each category.
    Use this for any question about menu prices, food details, or what's available.

    Args:
        category: Optional. One of "breakfast", "starters", "mains", "desserts", "drinks".
                  If omitted, returns the full menu.

    Returns a FoodAgentResponse with:
      - answer:      plain-English summary
      - cost:        list of items with name and price
      - description: combined description text for the matching items
    """
    valid_categories = ["breakfast", "starters", "mains", "desserts", "drinks"]
    if category:
        category = category.lower()
        if category not in valid_categories:
            return FoodAgentResponse(
                answer=f"Invalid category. Please choose from: {', '.join(valid_categories)}",
                cost=[], description=""
            )
        items = get_details_category_all(category)
        if not items:
            return FoodAgentResponse(answer=f"No items found for {category}.", cost=[], description="")
        return FoodAgentResponse(
            answer=f"Here are the available {category}.",
            cost=[FoodDrinkResult(item_name=item["item_name"], price=str(item["price"])) for item in items],
            description=" ".join(item["description"] for item in items)
        )
    return FoodAgentResponse(
        answer=f"Please specify a category: {', '.join(valid_categories)}", cost=[], description=""
    )

@tool
def opening_hours_tool(day: Optional[str] = None) -> OpeningHoursResponse:
    """
    Use this tool to answer customer inquiries about restaurant opening hours,
    closing times, and any special notes such as kitchen closing early,
    breakfast, or brunch availability.

    Args:
        day: Optional. One of "Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday".
             If omitted, returns hours for the full week.

    Returns an OpeningHoursResponse with:
      - answer: plain-English summary
      - hours:  list of day(s) with opening time, closing time, and note
    """
    all_hours = get_details_opening_time_all()
    if day:
        day = day.capitalize()
        match = [r for r in all_hours if r["day"] == day]
        if not match:
            return OpeningHoursResponse(answer=f"No opening hours found for {day}.", hours=[])
        info = match[0]
        return OpeningHoursResponse(
            answer=f"On {day}, we're open from {info['opens']} to {info['closes']}.",
            hours=[OpeningHoursResult(day=info["day"], opens=info["opens"], closes=info["closes"], note=info["note"])]
        )
    return OpeningHoursResponse(
        answer="Here are our opening hours for the week.",
        hours=[OpeningHoursResult(day=r["day"], opens=r["opens"], closes=r["closes"], note=r["note"]) for r in all_hours]
    )

@tool
def table_reservation(first_name: str, last_name: str, year: int, month: int, day: int, time: str) -> ReservationAgentResponse:
    """
    Use this tool to create a reservation for a guest.
    You MUST collect ALL of the following before calling this tool:
    first name, last name, year, month, day, and time.
    Do NOT assume any field — ask for each one explicitly.
    Do NOT assume the year — always ask the guest for the year.
    DO NOT assume the month - always ask for the month
    DO NOT assume the date and time - always ask for the date and time
    Do NOT proceed with only a first name — always ask for the last name.
    If the guest gives a month name (e.g. "July"), convert it to an integer (7).
    If the guest gives a vague time (e.g. "evening"), ask for a specific time in HH:MM format.

    Args:
        first_name: First name of the guest — always ask explicitly, never assume
        last_name:  Last name of the guest — always ask explicitly, never assume
        year:       Year of the reservation, e.g. 2026 — always ask, never assume
        month:      Month as an integer, e.g. 7 for July
        day:        Day as an integer, e.g. 5
        time:       Time as a string in HH:MM format, e.g. "19:00"

    Returns a ReservationAgentResponse with:
      - answer:        confirmation message with the reservation ID
      - reservations:  list with the full booking details
    """
    reservation_id = create_reservation(first_name, last_name, year, month, day, time)
    return ReservationAgentResponse(
        answer=f"Reservation confirmed! Your booking reference is #{reservation_id}. See you on {day}/{month}/{year} at {time}, {first_name.title()}!",
        reservations=[ReservationResult(reservation_id=reservation_id, first_name=first_name,
                                        last_name=last_name, year=year, month=month,
                                        day=day, time=time, status="confirmed")]
    )

@tool
def check_reservation(first_name: str, last_name: str) -> ReservationAgentResponse:
    """
    Use this tool to check the status of an existing reservation for a guest.
    You MUST collect both first name AND last name before calling this tool.
    Do NOT assume or proceed with only one name — always ask for the last name explicitly.
    Do NOT assume the guest's last name from context or previous messages.
    Return the status of the reservation, 'confirmed' or 'cancelled'.

    Args:
        first_name: First name of the guest — always ask explicitly, never assume
        last_name:  Last name of the guest — always ask explicitly, never assume

    Returns a ReservationAgentResponse with:
      - answer:        confirmation message with the reservation ID
      - reservations:  list with the full booking details
    """
    results = check_reservation_db(first_name, last_name)
    if not results:
        return ReservationAgentResponse(
            answer=f"No reservation found for {first_name.title()} {last_name.title()}. Please double check the name.",
            reservations=[]
        )
    return ReservationAgentResponse(
        answer=f"Found {len(results)} reservation(s) for {first_name.title()} {last_name.title()}.",
        reservations=[ReservationResult(**r) for r in results]
    )

@tool
def cancel_reservation(first_name: str, last_name: str) -> ReservationAgentResponse:
    """
    Use this tool to cancel an existing reservation for a guest.
    You MUST collect both first name AND last name before calling this tool.
    Do NOT assume or proceed with only one name — always ask for the last name explicitly.
    Do NOT cancel without verbally confirming the full name with the guest first.
    Do NOT assume the cancellation is for the same person as a previous query in the conversation.
    Return a confirmation message of cancellation.

    Args:
        first_name: First name of the guest — always ask explicitly, never assume
        last_name:  Last name of the guest — always ask explicitly, never assume

    Returns a ReservationAgentResponse with:
      - answer:        cancellation confirmation message
      - reservations:  list with the updated booking details showing cancelled status
    """
    result = cancel_reservation_db(first_name, last_name)
    if not result:
        return ReservationAgentResponse(
            answer=f"No active reservation found for {first_name.title()} {last_name.title()}. Please double check the name.",
            reservations=[]
        )
    return ReservationAgentResponse(
        answer=f"Reservation for {first_name.title()} {last_name.title()} on {result['day']}/{result['month']}/{result['year']} at {result['time']} has been successfully cancelled.",
        reservations=[ReservationResult(**result)]
    )

# ══════════════════════════════════════════════════════════════════════════════
# SYSTEM PROMPT
# ══════════════════════════════════════════════════════════════════════════════
RESTAURANT_SYSTEM = """
You are Theo — the warm, sharp-eyed host of the restaurant, the kind of person
who remembers regulars by their order and treats every guest like they've
been coming for years.

You have access to five tools. Use them smartly:
- menu_prices         → guest asks about menu items, prices, ingredients, or what's available
- opening_hours_tool  → guest asks about hours, whether you're open, or special notes (kitchen closing, brunch, etc.)
- table_reservation   → create a reservation — requires first name, last name, year, month, day, and time
- check_reservation   → return the status of a reservation — requires first name and last name
- cancel_reservation  → cancel an existing reservation — requires first name and last name

Tool rules:
- NEVER guess prices, menu items, or hours. Always use the tool.
- Call the tool first, then weave the result into natural conversation.
- If the guest doesn't specify a category or day, call the tool with no
  argument to give them the full picture rather than asking unnecessarily.
- You can call multiple tools in one turn if needed.

Reservation rules — follow these without exception:
- NEVER call table_reservation without ALL six fields: first name, last name, year, month, day, and time.
- NEVER assume the year — always ask the guest explicitly.
- NEVER assume the month — always ask the guest explicitly.
- NEVER assume the date — always ask the guest explicitly.
- NEVER assume a time — always ask the guest explicitly.
- NEVER proceed with only a first name — always ask for the last name before calling any reservation tool.
- NEVER assume a last name from context or earlier in the conversation.
- If the guest gives a month name like "July", convert it to 7 before calling the tool.
- If the guest gives a vague time like "evening" or "around 7", ask for the exact time in HH:MM format.
- Before cancelling, always confirm the full name back to the guest and ask them to confirm before calling the tool.
- If a reservation is not found, tell the guest warmly and ask them to double check the name.

How to sound like Theo:
- Open with something inviting — the smell of the kitchen, the buzz of the
  room, the specials board, the clatter of plates.
  Never "Great choice!" or "Absolutely!" — banned.
- Weave tool results into flowing prose. Never dump raw prices, items, or
  hours as a list or table.
- Be honest. If the kitchen's about to close, say so. If a dish isn't great
  that night, say so. If hours are unusual on a given day, mention it.
- 3–6 sentences of flowing prose. No bullets. No headers. No lists.
- Close with a recommendation or a small question to help the guest decide.

Language rule: Always reply in the same language the guest wrote in.
"""

# ══════════════════════════════════════════════════════════════════════════════
# AGENT SETUP  (cached so it's built once per session)
# ══════════════════════════════════════════════════════════════════════════════
@st.cache_resource
def build_agent():
    tools = [menu_prices, opening_hours_tool, table_reservation, check_reservation, cancel_reservation]
    llm   = ChatOpenAI(temperature=0, model_name=MODEL)
    llm_with_tools = llm.bind_tools(tools)
    tool_node      = ToolNode(tools)

    def agent_reason(state: MessagesState) -> MessagesState:
        response = llm_with_tools.invoke([SystemMessage(content=RESTAURANT_SYSTEM), *state["messages"]])
        return {"messages": [response]}

    def should_continue(state: MessagesState) -> str:
        return "act" if state["messages"][-1].tool_calls else END

    graph = StateGraph(MessagesState)
    graph.add_node("agent_reason", agent_reason)
    graph.add_node("act", tool_node)
    graph.set_entry_point("agent_reason")
    graph.add_conditional_edges("agent_reason", should_continue, {"act": "act", END: END})
    graph.add_edge("act", "agent_reason")
    return graph.compile(checkpointer=MemorySaver())

def chat(app, question: str, session_id: str = "user_001") -> str:
    config   = {"configurable": {"thread_id": session_id}}
    response = app.invoke({"messages": [HumanMessage(content=question)]}, config=config)
    return response["messages"][-1].content

# ══════════════════════════════════════════════════════════════════════════════
# STREAMLIT UI
# ══════════════════════════════════════════════════════════════════════════════
st.set_page_config(page_title="Theo — Restaurant Host", page_icon="🍽️", layout="centered")

st.markdown("""
<style>
    .stChatMessage { border-radius: 12px; }
    .block-container { max-width: 720px; padding-top: 2rem; }
    h1 { font-size: 1.6rem; font-weight: 700; }
</style>
""", unsafe_allow_html=True)

st.title("🍽️ Theo — Your Restaurant Host")
st.caption("Ask about the menu, opening hours, or make a reservation.")

# ── API key ────────────────────────────────────────────────────────────────
if not os.environ.get("OPENAI_API_KEY"):
    st.error("OPENAI_API_KEY not found. Make sure your .env file exists in the same folder as this script.")
    st.stop()

# ── Init DB and agent ──────────────────────────────────────────────────────
init_db()
app = build_agent()

# ── Session state ──────────────────────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": "Good evening — the kitchen's in full swing and there's a great energy in here tonight. What can I do for you?"}
    ]

if "session_id" not in st.session_state:
    import uuid
    st.session_state.session_id = str(uuid.uuid4())

# ── Render chat history ────────────────────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# ── Chat input ─────────────────────────────────────────────────────────────
if prompt := st.chat_input("Ask Theo anything..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Theo is thinking..."):
            reply = chat(app, prompt, session_id=st.session_state.session_id)
        st.markdown(reply)

    st.session_state.messages.append({"role": "assistant", "content": reply})

# ── Sidebar ────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("### About Theo")
    st.markdown("Theo is the warm, sharp-eyed host of the restaurant. Ask him about the menu, opening hours, or to make, check, or cancel a reservation.")
    st.divider()
    st.markdown("**Available tools**")
    st.markdown("- 🍴 Menu & prices\n- 🕐 Opening hours\n- 📅 Make reservation\n- 🔍 Check reservation\n- ❌ Cancel reservation")
    st.divider()
    if st.button("🔄 New conversation"):
        st.session_state.messages = [
            {"role": "assistant", "content": "Good evening — the kitchen's in full swing and there's a great energy in here tonight. What can I do for you?"}
        ]
        import uuid
        st.session_state.session_id = str(uuid.uuid4())
        st.rerun()

Writing streamlit_app.py


In [ ]:
!streamlit run streamlit_app.py


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.178.59:8501

/opt/anaconda3/envs/tf-metal/lib/python3.11/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## DOCKER

In [1]:
import os

folders = ["data", "db", "tools", "agent"]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

for folder in ["data","db", "tools", "agent"]:
    init_path = os.path.join(folder, "__init__.py")
    if not os.path.exists(init_path):
        open(init_path, "w").close()

In [2]:
%%writefile data/menu_data.py

menu = {
    "breakfast": [
        {"item_name": "Full English Breakfast", "description": "Eggs, bacon, sausage, beans, mushrooms, tomato and toast", "price": 11.50},
        {"item_name": "Eggs Benedict", "description": "Poached eggs, ham and hollandaise on an English muffin", "price": 9.50},
        {"item_name": "Avocado Toast", "description": "Smashed avocado on toasted sourdough", "price": 8.50},
        {"item_name": "Pancake Stack", "description": "Fluffy pancakes with maple syrup and berries", "price": 8.00},
        {"item_name": "French Toast", "description": "Brioche toast served with cinnamon and syrup", "price": 8.50},
        {"item_name": "Omelette", "description": "Three-egg omelette with cheese, ham or vegetables", "price": 8.50},
    ],
    "starters": [
        {"item_name": "Garlic Bread", "description": "Toasted sourdough with garlic butter", "price": 4.50},
        {"item_name": "Soup of the Day", "description": "Ask your server for today's selection", "price": 6.00},
        {"item_name": "Bruschetta", "description": "Grilled bread, tomato, basil, olive oil", "price": 6.50},
        {"item_name": "Calamari", "description": "Lightly fried, served with lemon aioli", "price": 8.50},
    ],
    "mains": [
        {"item_name": "Grilled Sea Bass", "description": "With lemon butter and seasonal vegetables", "price": 18.50},
        {"item_name": "Mushroom Risotto", "description": "Arborio rice, parmesan, truffle oil", "price": 14.00},
        {"item_name": "Margherita Pizza", "description": "San Marzano tomato, mozzarella, basil", "price": 12.50},
        {"item_name": "Grilled Ribeye Steak", "description": "10oz, served with fries and peppercorn sauce", "price": 24.00},
        {"item_name": "Chicken Parmigiana", "description": "Breaded chicken breast, marinara, mozzarella", "price": 16.50},
        {"item_name": "Vegetable Curry", "description": "Seasonal vegetables in a coconut curry, served with rice", "price": 13.50},
    ],
    "desserts": [
        {"item_name": "Chocolate Fondant", "description": "Warm, with vanilla ice cream", "price": 7.50},
        {"item_name": "Tiramisu", "description": "Classic Italian, espresso-soaked sponge", "price": 6.50},
        {"item_name": "Cheesecake", "description": "New York style, berry compote", "price": 6.50},
        {"item_name": "Sticky Toffee Pudding", "description": "Rich sponge with toffee sauce", "price": 7.00},
    ],
    "drinks": [
        {"item_name": "House Red (175ml)", "description": "Rioja", "price": 5.50},
        {"item_name": "House White (175ml)", "description": "Sauvignon Blanc", "price": 5.50},
        {"item_name": "Draught Beer (Pint)", "description": "Local lager on tap", "price": 5.00},
        {"item_name": "Sparkling Water", "description": "500ml bottle", "price": 2.50},
        {"item_name": "Espresso", "description": "Double shot", "price": 2.80},
        {"item_name": "Latte", "description": "Smooth espresso with steamed milk", "price": 3.80},
    ],
}

Overwriting data/menu_data.py


In [3]:
%%writefile data/hours_data.py

opening_hours = {
    "Monday":    {"opens": "12:00", "closes": "21:00", "note": ""},
    "Tuesday":   {"opens": "12:00", "closes": "21:00", "note": ""},
    "Wednesday": {"opens": "12:00", "closes": "21:00", "note": ""},
    "Thursday":  {"opens": "12:00", "closes": "21:00", "note": ""},
    "Friday":    {"opens": "12:00", "closes": "23:30", "note": "Kitchen closes 30 min early"},
    "Saturday":  {"opens": "09:00", "closes": "23:30", "note": "Breakfast served until 12:00"},
    "Sunday":    {"opens": "09:00", "closes": "20:00", "note": "Brunch menu until 14:00"},
}

Overwriting data/hours_data.py


In [4]:
%%writefile db/database.py

import sqlite3
from data.menu_data import menu
from data.hours_data import opening_hours

DB = "db/restaurant.db"

def init_db():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('CREATE TABLE IF NOT EXISTS opening_hours (days TEXT, Open TEXT, close TEXT, note TEXT)')
        cursor.execute('CREATE TABLE IF NOT EXISTS menu (category TEXT, item_name TEXT, description TEXT, price REAL)')
        cursor.execute('''CREATE TABLE IF NOT EXISTS reservation (
            reservation_id INTEGER PRIMARY KEY AUTOINCREMENT,
            first_name TEXT, last_name TEXT,
            year INTEGER, month INTEGER, day INTEGER,
            time TEXT, status TEXT
        )''')
        if cursor.execute("SELECT COUNT(*) FROM opening_hours").fetchone()[0] == 0:
            for day, info in opening_hours.items():
                cursor.execute("INSERT INTO opening_hours (days, Open, close, note) VALUES (?,?,?,?)",
                               (day, info["opens"], info["closes"], info["note"]))
        if cursor.execute("SELECT COUNT(*) FROM menu").fetchone()[0] == 0:
            for category, items in menu.items():
                for item in items:
                    cursor.execute("INSERT INTO menu (category, item_name, description, price) VALUES (?,?,?,?)",
                                   (category, item["item_name"], item["description"], item["price"]))
        conn.commit()

def get_details_category_all(category):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM menu WHERE category = ?', (category.lower(),))
        results = cursor.fetchall()
        return [{"item_name": row[1], "description": row[2], "price": row[3]} for row in results]

def get_details_opening_time_all():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM opening_hours')
        results = cursor.fetchall()
        return [{"day": row[0], "opens": row[1], "closes": row[2], "note": row[3]} for row in results]

def create_reservation(first_name, last_name, year, month, day, time):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO reservation (first_name, last_name, year, month, day, time, status) VALUES (?,?,?,?,?,?,?)',
            (first_name.lower(), last_name.lower(), year, month, day, time, "confirmed")
        )
        conn.commit()
        return cursor.lastrowid

def check_reservation_db(first_name, last_name):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM reservation WHERE first_name = ? AND last_name = ?',
                       (first_name.lower(), last_name.lower()))
        results = cursor.fetchall()
        if not results:
            return []
        return [{"reservation_id": r[0], "first_name": r[1], "last_name": r[2],
                 "year": r[3], "month": r[4], "day": r[5], "time": r[6], "status": r[7]} for r in results]

def cancel_reservation_db(first_name, last_name):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('UPDATE reservation SET status = ? WHERE first_name = ? AND last_name = ?',
                       ("cancelled", first_name.lower(), last_name.lower()))
        conn.commit()
        if cursor.rowcount == 0:
            return None
        cursor.execute('SELECT * FROM reservation WHERE first_name = ? AND last_name = ?',
                       (first_name.lower(), last_name.lower()))
        row = cursor.fetchone()
        return {"reservation_id": row[0], "first_name": row[1], "last_name": row[2],
                "year": row[3], "month": row[4], "day": row[5], "time": row[6], "status": row[7]}

Overwriting db/database.py


In [5]:
%%writefile tools/schemas.py

from typing import List
from pydantic import BaseModel, Field

class FoodDrinkResult(BaseModel):
    item_name: str = Field(description="name of food or drink")
    price:     str = Field(description="Price as a string, e.g. '$14.50'")

class FoodAgentResponse(BaseModel):
    answer:      str                   = Field(description="The agent's answer to the query")
    cost:        List[FoodDrinkResult] = Field(default_factory=list)
    description: str                   = Field(description="food details")

class OpeningHoursResult(BaseModel):
    day:    str = Field(description="day of the week")
    opens:  str = Field(description="opening time")
    closes: str = Field(description="closing time")
    note:   str = Field(description="special note")

class OpeningHoursResponse(BaseModel):
    answer: str                      = Field(description="The agent's answer")
    hours:  List[OpeningHoursResult] = Field(default_factory=list)

class ReservationResult(BaseModel):
    reservation_id: int = Field(description="Unique booking reference number")
    first_name:     str = Field(description="First name of the guest")
    last_name:      str = Field(description="Last name of the guest")
    year:           int = Field(description="Year of the reservation")
    month:          int = Field(description="Month of the reservation")
    day:            int = Field(description="Day of the reservation")
    time:           str = Field(description="Reservation time")
    status:         str = Field(description="'confirmed' or 'cancelled'")

class ReservationAgentResponse(BaseModel):
    answer:       str                     = Field(description="The agent's plain-English answer")
    reservations: List[ReservationResult] = Field(default_factory=list)

Overwriting tools/schemas.py


In [6]:
%%writefile tools/menu_tool.py

from typing import Optional
from langchain_core.tools import tool
from tools.schemas import FoodDrinkResult, FoodAgentResponse
from db.database import get_details_category_all

@tool
def menu_prices(category: Optional[str] = None) -> FoodAgentResponse:
    """
    Use this tool to find restaurant menu items, prices, and descriptions.
    The guest states what they want and use this tool to provide varieties
    of what is offered based on each category.
    Use this for any question about menu prices, food details, or what's available.

    Args:
        category: Optional. One of "breakfast", "starters", "mains", "desserts", "drinks".
                  If omitted, returns the full menu.

    Returns a FoodAgentResponse with:
      - answer:      plain-English summary
      - cost:        list of items with name and price
      - description: combined description text for the matching items
    """
    valid_categories = ["breakfast", "starters", "mains", "desserts", "drinks"]
    if category:
        category = category.lower()
        if category not in valid_categories:
            return FoodAgentResponse(
                answer=f"Invalid category. Please choose from: {', '.join(valid_categories)}",
                cost=[], description=""
            )
        items = get_details_category_all(category)
        if not items:
            return FoodAgentResponse(answer=f"No items found for {category}.", cost=[], description="")
        return FoodAgentResponse(
            answer=f"Here are the available {category}.",
            cost=[FoodDrinkResult(item_name=item["item_name"], price=str(item["price"])) for item in items],
            description=" ".join(item["description"] for item in items)
        )
    return FoodAgentResponse(
        answer=f"Please specify a category: {', '.join(valid_categories)}", cost=[], description=""
    )

Overwriting tools/menu_tool.py


In [7]:
%%writefile tools/hours_tool.py

from typing import Optional
from langchain_core.tools import tool
from tools.schemas import OpeningHoursResult, OpeningHoursResponse
from db.database import get_details_opening_time_all

@tool
def opening_hours_tool(day: Optional[str] = None) -> OpeningHoursResponse:
    """
    Use this tool to answer customer inquiries about restaurant opening hours,
    closing times, and any special notes such as kitchen closing early,
    breakfast, or brunch availability.

    Args:
        day: Optional. One of "Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday".
             If omitted, returns hours for the full week.

    Returns an OpeningHoursResponse with:
      - answer: plain-English summary
      - hours:  list of day(s) with opening time, closing time, and note
    """
    all_hours = get_details_opening_time_all()
    if day:
        day = day.capitalize()
        match = [r for r in all_hours if r["day"] == day]
        if not match:
            return OpeningHoursResponse(answer=f"No opening hours found for {day}.", hours=[])
        info = match[0]
        return OpeningHoursResponse(
            answer=f"On {day}, we're open from {info['opens']} to {info['closes']}.",
            hours=[OpeningHoursResult(day=info["day"], opens=info["opens"], closes=info["closes"], note=info["note"])]
        )
    return OpeningHoursResponse(
        answer="Here are our opening hours for the week.",
        hours=[OpeningHoursResult(day=r["day"], opens=r["opens"], closes=r["closes"], note=r["note"]) for r in all_hours]
    )

Overwriting tools/hours_tool.py


In [8]:
%%writefile tools/reservation_tools.py

from langchain_core.tools import tool
from tools.schemas import ReservationResult, ReservationAgentResponse
from db.database import create_reservation, check_reservation_db, cancel_reservation_db

@tool
def table_reservation(first_name: str, last_name: str, year: int, month: int, day: int, time: str) -> ReservationAgentResponse:
    """
    Use this tool to create a reservation for a guest.
    You MUST collect ALL of the following before calling this tool:
    first name, last name, year, month, day, and time.
    Do NOT assume any field — ask for each one explicitly.
    Do NOT assume the year — always ask the guest for the year.
    Do NOT proceed with only a first name — always ask for the last name.
    If the guest gives a month name (e.g. "July"), convert it to an integer (7).
    If the guest gives a vague time (e.g. "evening"), ask for a specific time in HH:MM format.

    Args:
        first_name: First name of the guest — always ask explicitly, never assume
        last_name:  Last name of the guest — always ask explicitly, never assume
        year:       Year of the reservation, e.g. 2026 — always ask, never assume
        month:      Month as an integer, e.g. 7 for July
        day:        Day as an integer, e.g. 5
        time:       Time as a string in HH:MM format, e.g. "19:00"
    """
    reservation_id = create_reservation(first_name, last_name, year, month, day, time)
    return ReservationAgentResponse(
        answer=f"Reservation confirmed! Your booking reference is #{reservation_id}. See you on {day}/{month}/{year} at {time}, {first_name.title()}!",
        reservations=[ReservationResult(reservation_id=reservation_id, first_name=first_name,
                                        last_name=last_name, year=year, month=month,
                                        day=day, time=time, status="confirmed")]
    )

@tool
def check_reservation(first_name: str, last_name: str) -> ReservationAgentResponse:
    """
    Use this tool to check the status of an existing reservation for a guest.
    You MUST collect both first name AND last name before calling this tool.
    Do NOT assume or proceed with only one name — always ask for the last name explicitly.
    Do NOT assume the guest's last name from context or previous messages.

    Args:
        first_name: First name of the guest — always ask explicitly, never assume
        last_name:  Last name of the guest — always ask explicitly, never assume
    """
    results = check_reservation_db(first_name, last_name)
    if not results:
        return ReservationAgentResponse(
            answer=f"No reservation found for {first_name.title()} {last_name.title()}. Please double check the name.",
            reservations=[]
        )
    return ReservationAgentResponse(
        answer=f"Found {len(results)} reservation(s) for {first_name.title()} {last_name.title()}.",
        reservations=[ReservationResult(**r) for r in results]
    )

@tool
def cancel_reservation(first_name: str, last_name: str) -> ReservationAgentResponse:
    """
    Use this tool to cancel an existing reservation for a guest.
    You MUST collect both first name AND last name before calling this tool.
    Do NOT assume or proceed with only one name — always ask for the last name explicitly.
    Do NOT cancel without verbally confirming the full name with the guest first.
    Do NOT assume the cancellation is for the same person as a previous query in the conversation.

    Args:
        first_name: First name of the guest — always ask explicitly, never assume
        last_name:  Last name of the guest — always ask explicitly, never assume
    """
    result = cancel_reservation_db(first_name, last_name)
    if not result:
        return ReservationAgentResponse(
            answer=f"No active reservation found for {first_name.title()} {last_name.title()}. Please double check the name.",
            reservations=[]
        )
    return ReservationAgentResponse(
        answer=f"Reservation for {first_name.title()} {last_name.title()} on {result['day']}/{result['month']}/{result['year']} at {result['time']} has been successfully cancelled.",
        reservations=[ReservationResult(**result)]
    )

Overwriting tools/reservation_tools.py


In [9]:
%%writefile agent/prompts.py

RESTAURANT_SYSTEM = """
You are Theo — the warm, sharp-eyed host of the restaurant, the kind of person
who remembers regulars by their order and treats every guest like they've
been coming for years.

You have access to five tools. Use them smartly:
- menu_prices         → guest asks about menu items, prices, ingredients, or what's available
- opening_hours_tool  → guest asks about hours, whether you're open, or special notes (kitchen closing, brunch, etc.)
- table_reservation   → create a reservation — requires first name, last name, year, month, day, and time
- check_reservation   → return the status of a reservation — requires first name and last name
- cancel_reservation  → cancel an existing reservation — requires first name and last name

Tool rules:
- NEVER guess prices, menu items, or hours. Always use the tool.
- Call the tool first, then weave the result into natural conversation.
- If the guest doesn't specify a category or day, call the tool with no
  argument to give them the full picture rather than asking unnecessarily.
- You can call multiple tools in one turn if needed.

Reservation rules — follow these without exception:
- NEVER call table_reservation without ALL six fields: first name, last name, year, month, day, and time.
- NEVER assume the year — always ask the guest explicitly.
- NEVER assume the month  - always ask the guest explicitly
- ALWAYS ASK FOR THE MONTH, DAY AND TIME
- NEVER assume the date - always ask the guest explicitly
- NEVER assume a time, always ask the guest explicitly
- NEVER proceed with only a first name — always ask for the last name before calling any reservation tool.
- ALWAYS ASK FOR THE MONTH, DAY AND TIME
- NEVER assume a last name from context or earlier in the conversation.
- If the guest gives a month name like "July", convert it to 7 before calling the tool.
- If the guest gives a vague time like "evening" or "around 7", ask for the exact time in HH:MM format.
- Before cancelling, always confirm the full name back to the guest and ask them to confirm before calling the tool.
- If a reservation is not found, tell the guest warmly and ask them to double check the name.

How to sound like Theo:
- Open with something inviting — the smell of the kitchen, the buzz of the
  room, the specials board, the clatter of plates.
  Never "Great choice!" or "Absolutely!" — banned.
- Weave tool results into flowing prose. Never dump raw prices, items, or
  hours as a list or table.
- Be honest. If the kitchen's about to close, say so. If a dish isn't great
  that night, say so. If hours are unusual on a given day, mention it.
- 3–6 sentences of flowing prose. No bullets. No headers. No lists.
- Close with a recommendation or a small question to help the guest decide.

Language rule: Always reply in the same language the guest wrote in.
"""

Overwriting agent/prompts.py


In [10]:
%%writefile agent/agent.py

import streamlit as st
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, MessagesState, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

from tools.menu_tool import menu_prices
from tools.hours_tool import opening_hours_tool
from tools.reservation_tools import table_reservation, check_reservation, cancel_reservation
from agent.prompts import RESTAURANT_SYSTEM

tools = [menu_prices, opening_hours_tool, table_reservation, check_reservation, cancel_reservation]

@st.cache_resource
def build_agent():
    llm            = ChatOpenAI(temperature=0, model_name="gpt-4.1-nano")
    llm_with_tools = llm.bind_tools(tools)
    tool_node      = ToolNode(tools)

    def agent_reason(state: MessagesState) -> MessagesState:
        response = llm_with_tools.invoke([SystemMessage(content=RESTAURANT_SYSTEM), *state["messages"]])
        return {"messages": [response]}

    def should_continue(state: MessagesState) -> str:
        return "act" if state["messages"][-1].tool_calls else END

    graph = StateGraph(MessagesState)
    graph.add_node("agent_reason", agent_reason)
    graph.add_node("act", tool_node)
    graph.set_entry_point("agent_reason")
    graph.add_conditional_edges("agent_reason", should_continue, {"act": "act", END: END})
    graph.add_edge("act", "agent_reason")
    return graph.compile(checkpointer=MemorySaver())

def chat(app, question: str, session_id: str) -> str:
    config   = {"configurable": {"thread_id": session_id}}
    response = app.invoke({"messages": [HumanMessage(content=question)]}, config=config)
    return response["messages"][-1].content

Overwriting agent/agent.py


In [11]:
%%writefile app.py

import os
import uuid
import streamlit as st
from dotenv import load_dotenv
from db.database import init_db
from agent.agent import build_agent, chat

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    st.error("OPENAI_API_KEY not found. Make sure your .env file exists.")
    st.stop()

st.set_page_config(page_title="Theo — Restaurant Host", page_icon="🍽️", layout="centered")
st.markdown("""
<style>
    .block-container { max-width: 720px; padding-top: 2rem; }
</style>
""", unsafe_allow_html=True)

st.title("🍽️ Theo — Your Restaurant Host")
st.caption("Ask about the menu, opening hours, or make a reservation.")

init_db()
app = build_agent()

if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": "Good evening — the kitchen's in full swing and there's a great energy in here tonight. What can I do for you?"}
    ]
if "session_id" not in st.session_state:
    st.session_state.session_id = str(uuid.uuid4())

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Ask Theo anything..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)
    with st.chat_message("assistant"):
        with st.spinner("Theo is thinking..."):
            reply = chat(app, prompt, session_id=st.session_state.session_id)
        st.markdown(reply)
    st.session_state.messages.append({"role": "assistant", "content": reply})

with st.sidebar:
    st.markdown("### About Theo")
    st.markdown("Theo is your restaurant host. Ask him about the menu, hours, or reservations.")
    st.divider()
    st.markdown("**Tools**")
    st.markdown("- 🍴 Menu & prices\n- 🕐 Opening hours\n- 📅 Make reservation\n- 🔍 Check reservation\n- ❌ Cancel reservation")
    st.divider()
    if st.button("🔄 New conversation"):
        st.session_state.messages = [
            {"role": "assistant", "content": "Good evening — the kitchen's in full swing and there's a great energy in here tonight. What can I do for you?"}
        ]
        st.session_state.session_id = str(uuid.uuid4())
        st.rerun()

Overwriting app.py


In [12]:
%%writefile requirements.txt

streamlit
langchain
langchain-openai
langchain-community
langgraph
pydantic
python-dotenv
openai

Overwriting requirements.txt


In [13]:
%%writefile Dockerfile

# ── Base image ─────────────────────────────────────────────────────────────
FROM python:3.11-slim

# ── Working directory ──────────────────────────────────────────────────────
WORKDIR /app

# ── Install dependencies ───────────────────────────────────────────────────
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ── Copy project files ─────────────────────────────────────────────────────
COPY . .

# ── Create folders in case they don't exist ────────────────────────────────
#RUN mkdir -p db data tools agent

# ── Expose Streamlit port ──────────────────────────────────────────────────
EXPOSE 8501

# ── Run the app ────────────────────────────────────────────────────────────
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]

Overwriting Dockerfile


In [14]:
%%writefile .dockerignore

.env
__pycache__
*.pyc
*.pyo
db/restaurant.db
.ipynb_checkpoints
*.ipynb

Overwriting .dockerignore


In [75]:
# build
!docker build -t restaurant-bot .


[+] Building 0.0s (0/1)                                    docker:desktop-linux
[+] Building 0.2s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 1.59kB                                     0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.1s
[+] Building 0.3s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 1.59kB                                     0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.3s
[+] Building 0.5s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 1.59kB                                     0.0s
 => [internal] load metadata for docker

In [77]:
# run — inject the key at runtime from your .env to acess as docker ignore keeps the .env file from getting to the image
!docker run -p 8501:8501 --env-file .env restaurant-bot



2026-06-24 12:20:45.231 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.17.0.2:8501
  External URL: http://92.208.182.234:8501

^C
  Stopping...
